In [3]:

import numpy as np 
import pandas as pd

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub

/kaggle/input/datasets/arashnic/book-recommendation-dataset/Ratings.csv
/kaggle/input/datasets/arashnic/book-recommendation-dataset/Users.csv
/kaggle/input/datasets/arashnic/book-recommendation-dataset/classicRec.png
/kaggle/input/datasets/arashnic/book-recommendation-dataset/Books.csv
/kaggle/input/datasets/arashnic/book-recommendation-dataset/DeepRec.png
/kaggle/input/datasets/arashnic/book-recommendation-dataset/recsys_taxonomy2.png


In [12]:

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import pickle
books= pd.read_csv('/kaggle/input/datasets/arashnic/book-recommendation-dataset/Books.csv')
ratings = pd.read_csv('/kaggle/input/datasets/arashnic/book-recommendation-dataset/Ratings.csv')
books.head()



/tmp/ipykernel_58/230672467.py:7: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  books= pd.read_csv('/kaggle/input/datasets/arashnic/book-recommendation-dataset/Books.csv')


,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...


In [13]:
ratings.head()

,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6


In [31]:
users= pd.read_csv('/kaggle/input/datasets/arashnic/book-recommendation-dataset/Users.csv')
users.head()

,User-ID,Location,Age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0
2,3,"moscow, yukon territory, russia",NaN
3,4,"porto, v.n.gaia, portugal",17.0
4,5,"farnborough, hants, united kingdom",NaN


In [14]:
books = books[['ISBN', 'Book-Title', 'Book-Author', 'Publisher', 'Year-Of-Publication', 'Image-URL-M']]
books.dropna(inplace=True)
books.drop_duplicates(subset='Book-Title', keep='first', inplace=True)

books['tags'] = books['Book-Author'].astype(str) + ' ' + books['Publisher'].astype(str) + ' ' + books['Year-Of-Publication'].astype(str)
books['tags'] = books['tags'].str.lower()

books.reset_index(drop=True, inplace=True)
books.head(3)

,ISBN,Book-Title,Book-Author,Publisher,Year-Of-Publication,Image-URL-M,tags
0,0195153448,Classical Mythology,Mark P. O. Morford,Oxford University Press,2002,http://images.amazon.com/images/P/0195153448.0...,mark p. o. morford oxford university press 2002
1,0002005018,Clara Callan,Richard Bruce Wright,HarperFlamingo Canada,2001,http://images.amazon.com/images/P/0002005018.0...,richard bruce wright harperflamingo canada 2001
2,0060973129,Decision in Normandy,Carlo D'Este,HarperPerennial,1991,http://images.amazon.com/images/P/0060973129.0...,carlo d'este harperperennial 1991


In [17]:
books.shape

(242132, 7)

In [18]:
# Select top 15,000 books to prevent Kaggle RAM memory crashes
df = books.head(15000).copy()

tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
vector_matrix = tfidf.fit_transform(df['tags'])

similarity_matrix = cosine_similarity(vector_matrix)
print(f"Similarity matrix shape: {similarity_matrix.shape}")

Similarity matrix shape: (15000, 15000)


In [19]:
def recommend(book_name):
    if book_name not in df['Book-Title'].values:
        return "Book not found in the dataset."
    
    index = df[df['Book-Title'] == book_name].index[0]
    distances = sorted(list(enumerate(similarity_matrix[index])), key=lambda x: x[1], reverse=True)[1:6]
    
    recommended_books = []
    for i in distances:
        item = []
        temp_df = df[df.index == i[0]]
        item.extend(list(temp_df['Book-Title'].values))
        item.extend(list(temp_df['Book-Author'].values))
        item.extend(list(temp_df['Image-URL-M'].values))
        recommended_books.append(item)
        
    return recommended_books

# Test with a book title present in df
recommend(df['Book-Title'].iloc[0])

[['One Nation, Underprivileged: Why American Poverty Affects Us All',
  'Mark Robert Rank',
  'http://images.amazon.com/images/P/0195101685.01.MZZZZZZZ.jpg'],
 ['Metaphysical Lyrics Poems 17 Cen',
  'Grierson',
  'http://images.amazon.com/images/P/0198811020.01.MZZZZZZZ.jpg'],
 ['Facial Justice (Twentieth Century Classics)',
  'L. P. Hartley',
  'http://images.amazon.com/images/P/0192820575.01.MZZZZZZZ.jpg'],
 ['Darwinizing Culture: The Status of Memetics As a Science',
  'Robert Aunger',
  'http://images.amazon.com/images/P/0192632442.01.MZZZZZZZ.jpg'],
 ['The Complete Poetry of Catullus (Wisconsin Studies in Classics)',
  'Gaius Valerius Catullus',
  'http://images.amazon.com/images/P/0299177742.01.MZZZZZZZ.jpg']]

In [30]:
import pickle
from scipy.sparse import save_npz

# 1. Save books dictionary (~5-10 MB)
pickle.dump(df.to_dict(), open('book_dict.pkl', 'wb'))

# 2. Save the sparse TF-IDF matrix (~2-5 MB instead of 1.8 GB)
save_npz('vector_matrix.npz', vector_matrix)